# Data cleaning and preparation

Run every cell from top to bottom. This notebook contains all quality checks, cleaning, the training/validation split, rare-category grouping, encoding, and verification in one place. It requires pandas and NumPy.

Original CSVs in `data_sets/` are preserved. Results are saved in `data_processed/ml_categorical/`: `train.csv`, `validation.csv`, `test.csv`, and `preprocessing_state.json`. Check reports and intermediate tables appear here instead of being saved as separate files.

The existing association notebook remains a separate descriptive analysis of original observed answers. No model is trained here.


## 1. Load the original data and define valid survey responses

The labeled training data have two vaccination outcomes; the supplied test data have no observed outcomes. Numeric survey codes are categorical answers, including capped household counts. Allowed values follow the [DrivenData data description](https://www.drivendata.org/competitions/66/flushot-learning/page/211/).


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'data_sets/training_set_features.csv').is_file()
             and (p / 'data_sets/training_set_labels.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the project or notebook folder.')
OUT = ROOT / 'data_processed/ml_categorical'

raw = pd.read_csv(ROOT / 'data_sets/training_set_features.csv')
labels = pd.read_csv(ROOT / 'data_sets/training_set_labels.csv')
test = pd.read_csv(ROOT / 'data_sets/test_set_features.csv')
ID = 'respondent_id'
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
SEED = 42
VALIDATION_FRACTION = 0.2
RARE_FRACTION = 0.01  # Baseline choice; tune using training CV, not the test set.
HIGH_CARDINALITY = ['employment_industry', 'employment_occupation']
UNKNOWN = 'Unknown'
OTHER = '__Other__'  # Rare or previously unseen categories; distinct from missing.
BINARY = [
    'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask',
    'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home',
    'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
    'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance',
]
OPINIONS = [f'opinion_{v}_{q}' for v in ['h1n1', 'seas']
            for q in ['vacc_effective', 'risk', 'sick_from_vacc']]
NUMERIC_DOMAINS = {
    **{c: [0, 1] for c in BINARY},
    **{c: [1, 2, 3, 4, 5] for c in OPINIONS},
    'h1n1_concern': [0, 1, 2, 3], 'h1n1_knowledge': [0, 1, 2],
    'household_adults': [0, 1, 2, 3], 'household_children': [0, 1, 2, 3],
}
TEXT_DOMAINS = {
    'age_group': ['18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'],
    'education': ['< 12 Years', '12 Years', 'Some College', 'College Graduate'],
    'race': ['Black', 'Hispanic', 'Other or Multiple', 'White'],
    'sex': ['Female', 'Male'],
    'income_poverty': ['Below Poverty', '<= $75,000, Above Poverty', '> $75,000'],
    'marital_status': ['Married', 'Not Married'], 'rent_or_own': ['Own', 'Rent'],
    'employment_status': ['Employed', 'Not in Labor Force', 'Unemployed'],
    'census_msa': ['MSA, Not Principal City', 'MSA, Principal City', 'Non-MSA'],
}
OPAQUE = ['hhs_geo_region', *HIGH_CARDINALITY]


display(pd.DataFrame({'dataset': ['training features', 'training labels', 'test features'],
                      'rows': [len(raw), len(labels), len(test)],
                      'columns': [len(raw.columns), len(labels.columns), len(test.columns)]}))


             dataset   rows  columns
0  training features  26707       36
1    training labels  26707        3
2      test features  26708       36


## 2. Check duplicates and consistent respondent IDs

Remove only exact repeated records, including their IDs. Conflicting rows for the same ID stop preparation for investigation. Different people with identical answers are reported and retained. Never remove test respondents, because predictions must preserve their original order.


In [2]:
duplicate_report = pd.DataFrame([
    {'dataset': name, 'repeated_ID_rows': int(frame[ID].duplicated(keep=False).sum()),
     'extra_exact_records': int(frame.duplicated().sum()),
     'identical_answer_rows': int(frame.drop(columns=ID).duplicated(keep=False).sum())}
    for name, frame in [('training features', raw), ('test features', test)]
])
display(duplicate_report)
removed_feature_duplicates = int(raw.duplicated().sum())
removed_label_duplicates = int(labels.duplicated().sum())
raw = raw.drop_duplicates().reset_index(drop=True)
labels = labels.drop_duplicates().reset_index(drop=True)
for name, frame in [('training features', raw), ('labels', labels), ('test features', test)]:
    if frame[ID].isna().any() or not frame[ID].is_unique:
        raise ValueError(f'{name}: missing IDs or conflicting repeated IDs require investigation.')
assert set(labels.columns) == {ID, *TARGETS}, 'Unexpected label schema'
assert set(raw[ID]) == set(labels[ID]), 'Feature and label IDs do not match'
assert list(raw.columns) == list(test.columns), 'Train and test feature schemas differ'
assert not set(raw[ID]) & set(test[ID]), 'Labeled and test IDs overlap'
print(f'Exact repeated records removed: features={removed_feature_duplicates}, labels={removed_label_duplicates}')
print('ID, label alignment and feature schema checks passed.')


             dataset  repeated_ID_rows  extra_exact_records  identical_answer_rows
0  training features                 0                    0                      0
1      test features                 0                    0                      0
Exact repeated records removed: features=0, labels=0
ID, label alignment and feature schema checks passed.


## 3. Check targets and remove rows with missing outcomes

Align labels by ID, rather than assuming that the CSV row orders match. Remove labeled respondents with a missing vaccination outcome; reject other target values outside 0/1. No test labels are invented.


In [3]:
labels = labels.set_index(ID).loc[raw[ID], TARGETS].reset_index()
missing_target = labels[TARGETS].isna().any(axis=1)
removed_missing_targets = int(missing_target.sum())
raw = raw.loc[~missing_target].reset_index(drop=True)
labels = labels.loc[~missing_target].reset_index(drop=True)
assert labels[TARGETS].isin([0, 1]).all().all(), 'Targets must be complete binary values'
labels[TARGETS] = labels[TARGETS].astype(int)
print(f'Respondents removed for missing outcomes: {removed_missing_targets}')
display(labels[TARGETS].agg(['count', 'mean']))


Respondents removed for missing outcomes: 0
       h1n1_vaccine  seasonal_vaccine
count  26707.000000      26707.000000
mean       0.212454          0.465608


## 4. Inspect allowed values, outliers, text categories and missingness

Check numeric values against the permitted survey codes. Household counts of 3 mean “3 or more”; these are valid capped values, not outliers to trim. Opinion codes are answers, so IQR-based outlier removal is inappropriate here. Inspect the category table for unexpected text; opaque region/job codes are checked for format, rather than assigned invented meanings.

Normalize whitespace and correct “Principle” to “Principal” in urbanicity labels. These fixed rules do not learn from validation or test respondents.


In [4]:
def normalize_text(frame):
    out = frame.copy()
    for c in [*TEXT_DOMAINS, *OPAQUE]:
        out[c] = out[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    out['census_msa'] = out['census_msa'].str.replace('Principle', 'Principal', regex=False)
    return out

def audit_features(frame, name):
    expected = {ID, *NUMERIC_DOMAINS, *TEXT_DOMAINS, *OPAQUE}
    if set(frame.columns) != expected:
        raise ValueError(f'{name}: unexpected feature schema.')
    if frame[ID].isna().any() or not frame[ID].is_unique:
        raise ValueError(f'{name}: missing or conflicting duplicate IDs.')
    normalized = normalize_text(frame)
    records = []
    for col, allowed in {**NUMERIC_DOMAINS, **TEXT_DOMAINS}.items():
        values = pd.to_numeric(frame[col], errors='coerce') if col in NUMERIC_DOMAINS else normalized[col]
        invalid = frame[col].notna() & ~values.isin(allowed)
        records.append({'dataset': name, 'feature': col,
                        'allowed_values': json.dumps(allowed),
                        'invalid_count': int(invalid.sum()),
                        'observed_min': float(values.min()) if col in NUMERIC_DOMAINS else None,
                        'observed_max': float(values.max()) if col in NUMERIC_DOMAINS else None})
        if invalid.any():
            raise ValueError(f'{name}: invalid values in {col}: {frame.loc[invalid, col].unique()}')
    for col in OPAQUE:
        invalid = normalized[col].notna() & ~normalized[col].str.fullmatch(r'[a-z]{8}', na=False)
        if invalid.any():
            raise ValueError(f'{name}: unexpected opaque category format in {col}.')
    return records

value_checks = pd.DataFrame(audit_features(raw, 'labeled') + audit_features(test, 'test'))
display(value_checks)
print('Invalid or out-of-range values:', int(value_checks.invalid_count.sum()))
category_overview = []
for name, frame in [('labeled', raw), ('test', test)]:
    normalized = normalize_text(frame)
    for col in [*TEXT_DOMAINS, *OPAQUE]:
        category_overview.append({'dataset': name, 'feature': col,
                                 'categories': normalized[col].dropna().unique().tolist(),
                                 'category_count': int(normalized[col].nunique())})
display(pd.DataFrame(category_overview))
missing_report = pd.concat([
    pd.DataFrame({'dataset': name, 'feature': frame.columns,
                  'missing_count': frame.isna().sum().values,
                  'missing_percent': frame.isna().mean().values * 100})
    for name, frame in [('labeled', raw.drop(columns=ID)), ('test', test.drop(columns=ID))]
], ignore_index=True)
display(missing_report.sort_values(['dataset', 'missing_percent'], ascending=[True, False]))


    dataset                      feature                                                                     allowed_values  invalid_count  observed_min  observed_max
0   labeled    behavioral_antiviral_meds                                                                             [0, 1]              0           0.0           1.0
1   labeled         behavioral_avoidance                                                                             [0, 1]              0           0.0           1.0
2   labeled         behavioral_face_mask                                                                             [0, 1]              0           0.0           1.0
3   labeled        behavioral_wash_hands                                                                             [0, 1]              0           0.0           1.0
4   labeled  behavioral_large_gatherings                                                                             [0, 1]              0           0.0           1.

## 5. Preserve an 80/20 training/validation split

Split within each of the four vaccination-outcome combinations using seed 42. The supplied test file stays separate. All learned category frequencies and encoding vocabularies use only the training partition. This reproduces the previous split when source data are unchanged.

After fixing the split, exclude **training-partition rows only** with more than 20% unknown predictors (eight or more of 35). Missing occupation and industry are not counted as unknown when employment status confirms non-employment. Keep the full validation set for an honest assessment of all respondents, and preserve all test rows. The original 80/20 assignment stays fixed; filtering reduces the training portion afterward.


In [5]:
def split_indices(labels):
    rng = np.random.default_rng(SEED)
    training, validation = [], []
    for _, group in labels.groupby(TARGETS, sort=True):
        positions = rng.permutation(group.index.to_numpy())
        n_valid = int(round(len(positions) * VALIDATION_FRACTION))
        if not 0 < n_valid < len(positions):
            raise ValueError('Every outcome combination needs training and validation rows.')
        validation.extend(positions[:n_valid])
        training.extend(positions[n_valid:])
    return np.sort(training), np.sort(validation)

train_idx_before_filter, valid_idx = split_indices(labels)

def unknown_answer_counts(frame):
    mask = frame.drop(columns=ID).isna()
    not_employed = frame['employment_status'].isin(['Not in Labor Force', 'Unemployed'])
    for col in HIGH_CARDINALITY:
        mask.loc[not_employed, col] = False
    return mask.sum(axis=1)

UNKNOWN_FRACTION_LIMIT = 0.20
unknown_counts = unknown_answer_counts(raw)
excluded_train_idx = train_idx_before_filter[
    (unknown_counts.loc[train_idx_before_filter] / 35 > UNKNOWN_FRACTION_LIMIT).to_numpy()
]
train_idx = train_idx_before_filter[
    (unknown_counts.loc[train_idx_before_filter] / 35 <= UNKNOWN_FRACTION_LIMIT).to_numpy()
]
assert not set(excluded_train_idx) & set(valid_idx)
assert (unknown_counts.loc[train_idx] <= 7).all()
assert (unknown_counts.loc[excluded_train_idx] >= 8).all()
print(f'Training respondents excluded: {len(excluded_train_idx):,} of {len(train_idx_before_filter):,}')
print('Validation and test respondents are retained regardless of unknown percentage.')
assert not set(train_idx) & set(valid_idx)
assert set(train_idx) | set(excluded_train_idx) | set(valid_idx) == set(raw.index)
display(pd.DataFrame([
    {'dataset': name, 'rows': len(idx), **labels.loc[idx, TARGETS].mean().to_dict()}
    for name, idx in [('train', train_idx), ('validation', valid_idx)]
]))
print('Separate test respondents:', len(test))


Training respondents excluded: 1,002 of 21,366
Validation and test respondents are retained regardless of unknown percentage.
      dataset   rows  h1n1_vaccine  seasonal_vaccine
0       train  20364      0.213956          0.469554
1  validation   5341      0.212320          0.465643
Separate test respondents: 26708


### Unknown answers per column after filtering training rows

Compare the **same training partition before filtering (21,366 respondents)** with the **retained training rows (20,364 respondents)**. Count unknown answers before their conversion to the explicit `Unknown` category. Exclude occupation/industry gaps confirmed as not applicable; report those separately. Percentages use the row count of their respective partition. All 35 predictors are listed, including those with zero unknowns.

These counts describe remaining unanswered questions, even though the final encoded CSV contains no literal NaN values. Validation and test were not filtered and are not included in this comparison. No data are modified by this block.


In [6]:
def unknown_and_not_applicable_masks(frame):
    blank = frame.drop(columns=ID).isna()
    not_applicable = pd.DataFrame(False, index=frame.index, columns=blank.columns)
    known_not_employed = frame['employment_status'].isin(['Not in Labor Force', 'Unemployed'])
    for col in ['employment_industry', 'employment_occupation']:
        not_applicable[col] = blank[col] & known_not_employed
    return blank & ~not_applicable, not_applicable

training_before_filter = raw.loc[train_idx_before_filter]
training_after_filter = raw.loc[train_idx]
unknown_before, not_applicable_before = unknown_and_not_applicable_masks(training_before_filter)
unknown_after, not_applicable_after = unknown_and_not_applicable_masks(training_after_filter)
remaining_unknowns_by_column = pd.DataFrame({
    'unknown_before': unknown_before.sum(),
    'unknown_before_percent': unknown_before.mean().mul(100).round(2),
    'unknown_after': unknown_after.sum(),
    'unknown_after_percent': unknown_after.mean().mul(100).round(2),
    'unknown_answers_removed': unknown_before.sum() - unknown_after.sum(),
    'not_applicable_after': not_applicable_after.sum(),
}).sort_values(['unknown_after', 'unknown_before'], ascending=False)
remaining_unknowns_by_column.index.name = 'feature'
assert len(remaining_unknowns_by_column) == 35
assert remaining_unknowns_by_column['unknown_after'].sum() == unknown_answer_counts(training_after_filter).sum()
assert (remaining_unknowns_by_column['unknown_answers_removed'] >= 0).all()
assert (unknown_after.sum(axis=1) <= 7).all()
assert not (unknown_after & not_applicable_after).any().any()
print(f'Training rows: {len(training_before_filter):,} before; {len(training_after_filter):,} after filtering')
print(f'Unknown cells: {int(unknown_before.sum().sum()):,} before; {int(unknown_after.sum().sum()):,} after filtering')
print('Not-applicable employment gaps are excluded from unknown counts.')
with pd.option_context('display.max_rows', 35, 'display.max_columns', None):
    display(remaining_unknowns_by_column)


Training rows: 21,366 before; 20,364 after filtering
Unknown cells: 30,004 before; 17,301 after filtering
Not-applicable employment gaps are excluded from unknown counts.
                             unknown_before  unknown_before_percent  unknown_after  unknown_after_percent  unknown_answers_removed  not_applicable_after
feature                                                                                                                                                 
health_insurance                       9856                   46.13           8856                  43.49                     1000                     0
income_poverty                         3548                   16.61           2548                  12.51                     1000                     0
doctor_recc_h1n1                       1727                    8.08           1534                   7.53                      193                     0
doctor_recc_seasonal                   1727                    8

## 6. Clean the 35 predictors without median imputation

Treat binary answers, opinion scores, age groups and capped counts as categories. Use `Unknown` for missing answers. For respondents known to be unemployed or outside the labor force, missing occupation/industry becomes `not_employed`; unknown employment status does not establish unemployment.

No separate missing-indicator columns or `n_missing` feature are needed for this baseline. IDs and outcomes are not predictors. Cleaned tables stay available in memory as `clean_train`, `clean_validation`, and `clean_test`.


In [7]:
def clean_features(frame):
    """All 35 predictors become categories; no medians or extra indicators."""
    out = normalize_text(frame).drop(columns=ID)
    not_employed = out['employment_status'].isin(['Not in Labor Force', 'Unemployed'])
    for col in HIGH_CARDINALITY:
        out.loc[out[col].isna() & not_employed, col] = 'not_employed'
    for col in NUMERIC_DOMAINS:
        out[col] = pd.to_numeric(out[col]).astype('Int64').astype('string')
    out['health_insurance'] = out['health_insurance'].map({'0': 'Uninsured', '1': 'Insured'})
    return out.astype('string').fillna(UNKNOWN)

clean = clean_features(raw)
clean_train = clean.loc[train_idx].copy()
clean_validation = clean.loc[valid_idx].copy()
clean_test = clean_features(test)
frames = {'train': clean_train, 'validation': clean_validation, 'test': clean_test}
display(pd.DataFrame([
    {'dataset': name, 'rows': len(frame), 'predictors': frame.shape[1],
     'missing_cells': int(frame.isna().sum().sum())}
    for name, frame in frames.items()
]))
display(clean_train.head())


      dataset   rows  predictors  missing_cells
0       train  20364          35              0
1  validation   5341          35              0
2        test  26708          35              0
  h1n1_concern h1n1_knowledge behavioral_antiviral_meds behavioral_avoidance behavioral_face_mask behavioral_wash_hands behavioral_large_gatherings behavioral_outside_home behavioral_touch_face doctor_recc_h1n1 doctor_recc_seasonal chronic_med_condition child_under_6_months health_worker health_insurance opinion_h1n1_vacc_effective opinion_h1n1_risk opinion_h1n1_sick_from_vacc opinion_seas_vacc_effective opinion_seas_risk opinion_seas_sick_from_vacc      age_group         education   race     sex             income_poverty marital_status rent_or_own   employment_status hhs_geo_region               census_msa household_adults household_children employment_industry employment_occupation
0            1              0                         0                    0                    0                 

## 7. Group rare categories and fit compact encoding on training rows

For occupation and industry, categories representing less than 1% of retained training respondents are grouped as `__Other__`. Missing and not-employed responses remain separate. This baseline grouping is learned only from training responses.

**Binary questions:** use one 0/1 column, plus an unknown flag when missing answers occur in training. For example, `doctor_recc_h1n1=0` is no longer a separate column: the value column is 0 for No and 1 for Yes. A missing answer is 0 with its unknown flag set to 1.

**Other categorical questions:** keep a separate indicator for every observed category, including Black, Employed, Male, Rent, Not Married and not_employed. No reference category is omitted. This makes each category visible in the heatmap. Age/opinion encodings in this baseline remain categorical; notebook 04 applies the separately agreed ordered-score versions.

Do not reserve unused Unknown/Other columns. Missing categories are kept only when observed in training; Other exists only when job grouping actually creates it. Unexpected validation/future categories are explicitly rejected, except unseen nonmissing job categories can map to a learned Other group. Refitting/review is required for new missingness patterns; they are never silently treated as a real answer.


In [8]:
def fit_encoder(training):
    """Fit compact, reversible category encodings on training responses only."""
    minimum = int(np.ceil(len(training) * RARE_FRACTION))
    vocabulary, grouping, specs = {}, {}, {}
    numeric_binary = set(BINARY) - {'health_insurance'}
    text_binary_positive = {'health_insurance': 'Insured'}
    for col in training:
        counts = training[col].value_counts()
        rare = sorted([v for v, count in counts.items()
                       if col in HIGH_CARDINALITY and count < minimum
                       and v not in [UNKNOWN, 'not_employed']])
        grouping[col] = rare
        grouped = training[col].replace({v: OTHER for v in rare})
        levels = sorted(grouped.unique().tolist())
        regular = [level for level in levels if level not in [UNKNOWN, OTHER]]
        if not regular:
            raise ValueError(f'{col}: no observed non-missing category to encode.')
        vocabulary[col] = levels
        has_unknown = UNKNOWN in levels
        if col in numeric_binary or col in text_binary_positive:
            positive = '1' if col in numeric_binary else text_binary_positive[col]
            negative = '0' if col in numeric_binary else next(v for v in regular if v != positive)
            if set(regular) != {negative, positive}:
                raise ValueError(f'{col}: expected two observed binary categories.')
            columns = [col] + ([f'{col}_unknown'] if has_unknown else [])
            specs[col] = {'kind': 'binary', 'positive': positive, 'reference': negative,
                          'has_unknown': has_unknown, 'columns': columns}
        else:
            # Keep every observed category visible; omit no reference category.
            active = levels
            specs[col] = {'kind': 'full_categories', 'reference': None,
                          'active_levels': active, 'columns': [f'{col}={v}' for v in active]}
    return {'vocabulary': vocabulary, 'rare_categories': grouping, 'encoding': specs,
            'feature_columns_by_source': {col: spec['columns'] for col, spec in specs.items()},
            'rare_minimum_count': minimum, 'rare_fraction': RARE_FRACTION,
            'fit_source': 'training partition only', 'random_seed': SEED,
            'validation_fraction': VALIDATION_FRACTION,
            'unseen_policy': 'Known rare and unseen nonmissing job categories map to __Other__ only when learned in training; other unseen categories require review. Missing is never silently mapped to a real answer.'}


def transform(frame, state):
    grouped = pd.DataFrame(index=frame.index)
    encoded = {}
    for col, levels in state['vocabulary'].items():
        values = frame[col].replace({v: OTHER for v in state['rare_categories'][col]})
        unseen = ~values.isin(levels)
        if unseen.any():
            can_group = col in HIGH_CARDINALITY and OTHER in levels
            if not can_group or values.loc[unseen].eq(UNKNOWN).any():
                raise ValueError(f'{col}: category absent from training: {values.loc[unseen].unique().tolist()}')
            values = values.where(~unseen, OTHER)
        grouped[col] = values
        spec = state['encoding'][col]
        if spec['kind'] == 'binary':
            encoded[col] = values.eq(spec['positive']).astype(np.uint8)
            if spec['has_unknown']:
                encoded[f'{col}_unknown'] = values.eq(UNKNOWN).astype(np.uint8)
        else:
            for level in spec['active_levels']:
                encoded[f'{col}={level}'] = values.eq(level).astype(np.uint8)
    return grouped, pd.DataFrame(encoded, index=frame.index)


def decode_categories(encoded, state):
    """Recover grouped answers to verify that compact encoding loses no categories."""
    decoded = pd.DataFrame(index=encoded.index)
    for col, spec in state['encoding'].items():
        values = pd.Series(spec['reference'], index=encoded.index, dtype='string')
        if spec['kind'] == 'binary':
            if spec['has_unknown']:
                assert not (encoded[col].eq(1) & encoded[f'{col}_unknown'].eq(1)).any()
                values.loc[encoded[f'{col}_unknown'].eq(1)] = UNKNOWN
            values.loc[encoded[col].eq(1)] = spec['positive']
        else:
            assert encoded[spec['columns']].sum(axis=1).eq(1).all()
            for level, column in zip(spec['active_levels'], spec['columns']):
                values.loc[encoded[column].eq(1)] = level
        decoded[col] = values
    return decoded

state = fit_encoder(clean_train)
state['training_row_filter'] = {
    'applies_to': 'training partition only',
    'maximum_unknown_fraction': UNKNOWN_FRACTION_LIMIT,
    'denominator': 35,
    'exclude_confirmed_not_applicable_employment_gaps': True,
    'training_rows_before': len(train_idx_before_filter),
    'training_rows_after': len(train_idx),
    'excluded_respondent_ids': raw.loc[excluded_train_idx, ID].tolist(),
    'validation_unchanged': True,
    'test_unchanged': True,
}
encoded_frames = {}
for name, frame in frames.items():
    _, encoded_frames[name] = transform(frame, state)
state['encoded_columns'] = encoded_frames['train'].columns.tolist()
rare_report = []
for col in HIGH_CARDINALITY:
    for category, count in clean_train[col].value_counts().items():
        rare_report.append({'feature': col, 'category': category, 'training_count': int(count),
                            'grouped_as': OTHER if category in state['rare_categories'][col] else category})
print('Rare-category count threshold:', state['rare_minimum_count'])
display(pd.DataFrame(rare_report))
print('Encoded predictor columns:', len(state['encoded_columns']))

encoding_audit = pd.DataFrame([
    {'feature': col, 'encoding': spec['kind'], 'reference_value': spec['reference'],
     'output_columns': spec['columns'], 'number_of_columns': len(spec['columns'])}
    for col, spec in state['encoding'].items()
])
display(encoding_audit)


Rare-category count threshold: 204
                  feature      category  training_count    grouped_as
0     employment_industry  not_employed            9316  not_employed
1     employment_industry      fcxhlnwr            1992      fcxhlnwr
2     employment_industry      wxleyezf            1468      wxleyezf
3     employment_industry      ldnlellj             977      ldnlellj
4     employment_industry      pxcmvdjn             830      pxcmvdjn
5     employment_industry      atmlpfrs             721      atmlpfrs
6     employment_industry      arjwrbjb             706      arjwrbjb
7     employment_industry      xicduogh             669      xicduogh
8     employment_industry      mfikgejo             483      mfikgejo
9     employment_industry      vjjrobsf             430      vjjrobsf
10    employment_industry      rucpziij             417      rucpziij
11    employment_industry      xqicxuve             404      xqicxuve
12    employment_industry       Unknown             317

## 8. Verify the preparation

Check label alignment, coverage of labeled respondents across retained training, excluded training, and validation, identical encoded schemas, and exact recovery of grouped answers from the compact encodings. Also check that missing and unseen answers have different encodings, rare grouping cannot learn from validation rows, and invalid codes are rejected.


In [9]:
for name, encoded in encoded_frames.items():
    assert encoded.columns.tolist() == state['encoded_columns']
    assert not encoded.isna().any().any()
    assert encoded.isin([0, 1]).all().all()
    grouped, _ = transform(frames[name], state)
    decoded = decode_categories(encoded, state)
    pd.testing.assert_frame_equal(decoded, grouped.astype('string'))
    assert encoded.nunique().gt(1).all(), 'Unused/constant encoding column found'
    assert frames[name].shape[1] == 35 and not frames[name].isna().any().any()
for idx in [train_idx, valid_idx]:
    assert raw.loc[idx, ID].tolist() == labels.loc[idx, ID].tolist()
assert clean_test.index.tolist() == test.index.tolist()
assert fit_encoder(clean_train)['vocabulary'] == state['vocabulary']
a, b = split_indices(labels)
assert np.array_equal(a, train_idx_before_filter) and np.array_equal(b, valid_idx)
assert not set(train_idx) & set(excluded_train_idx)
# Unknown employment status does not prove non-employment.
unknown_fixture = raw.head(2).copy()
unknown_fixture.loc[:, 'employment_industry'] = None
unknown_fixture.loc[:, 'employment_occupation'] = None
unknown_fixture.loc[unknown_fixture.index[0], 'employment_status'] = 'Unemployed'
unknown_fixture.loc[unknown_fixture.index[1], 'employment_status'] = None
raw_counts = unknown_fixture.drop(columns=ID).isna().sum(axis=1)
adjusted_counts = unknown_answer_counts(unknown_fixture)
assert adjusted_counts.iloc[0] == raw_counts.iloc[0] - 2
assert adjusted_counts.iloc[1] == raw_counts.iloc[1]

# Missing is not the same as a real answer or an unseen category.
fixture = raw.head(3).copy()
fixture.loc[0, 'doctor_recc_h1n1'] = None
fixture.loc[1, 'doctor_recc_h1n1'] = 0
fixture.loc[0, 'employment_status'] = 'Unemployed'
fixture.loc[0, 'employment_industry'] = None
fixture.loc[1, 'employment_status'] = None
fixture.loc[1, 'employment_industry'] = None
fixture_clean = clean_features(fixture)
assert fixture_clean.loc[0, 'doctor_recc_h1n1'] == UNKNOWN
assert fixture_clean.loc[1, 'doctor_recc_h1n1'] == '0'
assert fixture_clean.loc[0, 'employment_industry'] == 'not_employed'
assert fixture_clean.loc[1, 'employment_industry'] == UNKNOWN
# Every binary answer (No, Yes, Unknown) has a distinct compact representation.
small = pd.DataFrame({'doctor_recc_h1n1': ['0', '1', UNKNOWN]})
small_state = fit_encoder(small)
grouped, encoded = transform(small, small_state)
assert encoded.columns.tolist() == ['doctor_recc_h1n1', 'doctor_recc_h1n1_unknown']
assert encoded.to_numpy().tolist() == [[0, 0], [1, 0], [0, 1]]
pd.testing.assert_frame_equal(decode_categories(encoded, small_state), grouped.astype('string'))
# The baseline keeps five indicators so every original age category is visible.
small = pd.DataFrame({'age_group': TEXT_DOMAINS['age_group']})
small_state = fit_encoder(small)
grouped, encoded = transform(small, small_state)
assert encoded.shape[1] == 5
assert encoded.sum(axis=1).eq(1).all()
pd.testing.assert_frame_equal(decode_categories(encoded, small_state), grouped.astype('string'))
# Complete nominal categories have separate columns, with no unused missing column.
small = pd.DataFrame({'sex': ['Male', 'Female']})
small_state = fit_encoder(small)
assert small_state['encoding']['sex']['columns'] == ['sex=Female', 'sex=Male']
try:
    transform(pd.DataFrame({'sex': [UNKNOWN]}), small_state)
except ValueError:
    pass
else:
    raise AssertionError('Previously unseen missingness must not be silently mapped to a real answer')
small = pd.DataFrame({'employment_industry': ['common'] * 298 + ['rare', UNKNOWN]})
small_state = fit_encoder(small)
assert small_state['rare_categories']['employment_industry'] == ['rare']
saved_state = json.dumps(small_state, sort_keys=True)
grouped, _ = transform(pd.DataFrame({'employment_industry': ['rare', 'unseen', UNKNOWN]}), small_state)
assert grouped.employment_industry.tolist() == [OTHER, OTHER, UNKNOWN]
assert json.dumps(small_state, sort_keys=True) == saved_state
# Every nominal category is explicitly represented, including previously omitted groups.
for source, spec in state['encoding'].items():
    if spec['kind'] == 'full_categories':
        assert spec['reference'] is None
        assert spec['active_levels'] == state['vocabulary'][source]
        assert len(spec['columns']) == len(state['vocabulary'][source])
assert 'race=Black' in state['encoded_columns']
assert 'employment_status=Employed' in state['encoded_columns']
assert 'employment_industry=not_employed' in state['encoded_columns']
assert 'employment_occupation=not_employed' in state['encoded_columns']
invalid = raw.head(3).copy()
invalid.loc[0, 'household_adults'] = 99
try:
    audit_features(invalid, 'fixture')
except ValueError as exc:
    assert 'household_adults' in str(exc)
else:
    raise AssertionError('An invalid survey code was not rejected')
print('All quality, alignment, split, encoding and leakage checks passed.')


All quality, alignment, split, encoding and leakage checks passed.


## 9. Save just the datasets needed for modeling and the encoding rules

Each output CSV contains `respondent_id` plus the encoded predictors. Training and validation also contain the two target columns. Keeping IDs and labels in the same table avoids many small companion files and preserves their relationship to each row. **Exclude IDs and targets from model inputs**, as shown below.

The JSON stores the fitted vocabulary, rare-category mappings, split settings and encoded column order. Keep it to transform future respondents consistently. It is not another check report. Detailed checks and cleaned previews remain in this notebook.


In [10]:
OUT.mkdir(parents=True, exist_ok=True)
prepared = {}
for name, encoded in encoded_frames.items():
    if name == 'test':
        table = pd.concat([test[[ID]], encoded.reset_index(drop=True)], axis=1)
    else:
        idx = train_idx if name == 'train' else valid_idx
        table = pd.concat([
            raw.loc[idx, [ID]].reset_index(drop=True),
            encoded.reset_index(drop=True),
            labels.loc[idx, TARGETS].reset_index(drop=True)
        ], axis=1)
    assert table[ID].is_unique and not table.isna().any().any()
    table.to_csv(OUT / f'{name}.csv', index=False)
    reloaded = pd.read_csv(OUT / f'{name}.csv')
    pd.testing.assert_frame_equal(reloaded, table, check_dtype=False)
    prepared[name] = table
(OUT / 'preprocessing_state.json').write_text(json.dumps(state, indent=2) + '\n')

# Use these objects in a later modeling notebook. No model is fitted here.
X_train = prepared['train'][state['encoded_columns']]
y_train = prepared['train'][TARGETS]
X_validation = prepared['validation'][state['encoded_columns']]
y_validation = prepared['validation'][TARGETS]
X_test = prepared['test'][state['encoded_columns']]
assert ID not in X_train and not set(TARGETS) & set(X_train.columns)
summary = pd.DataFrame([
    {'dataset': name, 'rows': len(table), 'encoded_predictors': len(state['encoded_columns']),
     'target_columns': 0 if name == 'test' else 2,
     'missing_cells': int(table.isna().sum().sum())}
    for name, table in prepared.items()
])
display(summary)
print('Saved three CSVs and one preprocessing JSON to:', OUT)


      dataset   rows  encoded_predictors  target_columns  missing_cells
0       train  20364                 164               2              0
1  validation   5341                 164               2              0
2        test  26708                 164               0              0
Saved three CSVs and one preprocessing JSON to: /Users/zahra.nourbakhsh/Downloads/ReDI/Vaccine_Prediction_Team_5/Vaccine_Prediction_Team_5/data_processed/ml_categorical


## Result

No duplicate respondents, missing vaccination outcomes, or invalid survey codes were found in the original data. The fixed training partition initially contains 21,366 respondents. Excluding its 1,002 rows with more than 20% unknown predictors leaves **20,364 training respondents**. Confirmed not-applicable occupation/industry gaps do not count toward the cutoff.

All **5,341 validation respondents** and **26,708 test respondents** are retained, including incomplete responses. The original raw CSVs and association notebook remain unchanged. Encoding is refitted on retained training rows only. The full validation population must be used to assess the effects of this filtering decision; model performance has not yet been evaluated.
